# CartPoleGame example

Create the env, step it by hand, look at the trajectory.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path
from typing import List, Tuple

# Walk up to the repo root so `core` and `games` import regardless of cwd
here = Path.cwd().resolve()
repo_root = next(p for p in [here, *here.parents] if (p / "core" / "game.py").exists())
sys.path.insert(0, str(repo_root))

import matplotlib.pyplot as plt
import numpy as np
from matplotlib import animation
from IPython.display import HTML

from games.cartpole.game import CartPoleGame, CartPoleState, LEFT, RIGHT

game = CartPoleGame(seed=0, render_mode="rgb_array")
game.state

## Step it by hand

A fixed action sequence — no policy. Alternating pushes cancel each other out instead of correcting the initial lean, so the pole keeps tipping the way it started and the episode ends about halfway through the sequence.

In [ ]:
actions = [RIGHT, LEFT] * 20

game.reset(seed=0)
states = [game.state]
frames = [game.render()]
for action in actions:
    state, reward, done, _ = game.step(action)
    states.append(state)
    frames.append(game.render())
    if done:
        break

print(f"{len(states) - 1} steps, score={game.total_score}, done={game.done}")

## The trajectory

In [ ]:
print(
    f"{'t':>3}  {'action':>6}  {'x':>8}  {'x_dot':>8}  {'theta':>8}  {'theta_dot':>9}"
)
for t, state in enumerate(states):
    action = f"{actions[t - 1]:>6}" if t else "     -"
    print(
        f"{t:>3}  {action}  {state.x:>+8.4f}  {state.x_dot:>+8.4f}  "
        f"{state.theta:>+8.4f}  {state.theta_dot:>+9.4f}"
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
series = [
    ([s.theta for s in states], "pole angle (rad)", 0.2095),
    ([s.x for s in states], "cart position", 2.4),
]
for ax, (values, label, limit) in zip(axes, series):
    ax.plot(values, marker=".")
    ax.axhline(limit, ls="--", c="r", lw=1)
    ax.axhline(-limit, ls="--", c="r", lw=1)
    ax.set_xlabel("step")
    ax.set_ylabel(label)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.axis("off")
image = ax.imshow(frames[0])
plt.close(fig)


def draw(i: int) -> Tuple[plt.Artist, ...]:
    image.set_data(frames[i])
    return (image,)


anim = animation.FuncAnimation(
    fig, draw, frames=len(frames), interval=1000 // 30, blit=True
)
HTML(anim.to_jshtml())